# Sensitivity budgets — MSM, Rosenbaum $\Gamma$, $c$-dependence, soft independence

Sweep quantitative budgets with `bound_path` / `bound_grid`, or fix one and call `bound`.


In [ ]:
from partialid import Assumptions, Query, Variables, bound, bound_grid, bound_path

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
base = Assumptions.manski("X", "Y")


## Tan MSM ($\Lambda$) and Rosenbaum $\Gamma$

Anchored Rosenbaum $\Gamma$ maps to Tan MSM with $\Lambda=\sqrt{\Gamma}$.


In [ ]:
print("=== Tan MSM path ===")
msm = bound_path(V, P, Q, base, param="Lambda", theta_min=1.0, theta_max=5.0)
print(msm.summary(at=3.0))

print("\n=== Rosenbaum Γ path ===")
gam = bound_path(V, P, Q, base, param="Gamma", theta_min=1.0, theta_max=9.0)
print(gam.summary(at=4.0))
# Γ=4 ↔ Λ=2
lo_g, hi_g, _ = gam.at(4.0)
lo_l, hi_l, _ = msm.at(2.0)
print(f"Γ=4 vs Λ=2 intervals match: {abs(lo_g - lo_l) < 1e-6 and abs(hi_g - hi_l) < 1e-6}")

r_g = bound(V, P, Q, base & Assumptions.rosenbaum_gamma("X", Gamma=4.0))
print(r_g.summary(Q))


## $c$-dependence (Masten–Poirier)

$\ell_\infty$ budget on the propensity; $c=0$ recovers no unobserved confounding.


In [ ]:
print("=== c-dependence path ===")
cpath = bound_path(V, P, Q, base, param="c", theta_min=0.0, theta_max=0.5, n_grid=6)
print(cpath.summary(at=0.1))
for c in (0.0, 0.1, 0.25, 0.5):
    lo, hi, w = cpath.at(c)
    print(f"  c={c:g}: [{lo:.4f}, {hi:.4f}]  width={w:.4f}")


## Soft independence (margin budget $\kappa$)

Allow $|S - S_0| \le \kappa$ instead of a hard identifying equality.


In [ ]:
print("=== margin budget ===")
r_soft = bound(V, P, Q, base & Assumptions.margin_budget(kappa=0.05))
print(r_soft.summary(Q))

kpath = bound_path(V, P, Q, base, param="kappa", theta_min=0.0, theta_max=0.2, n_grid=5)
print(kpath.summary(at=0.05))


## Dense grid with `bound_grid`

Returns one `BoundResult` per budget (useful when you want method / tightness
at each knot). Install `partialid[fast]` for persistent HiGHS warm-starts.


In [ ]:
print("=== bound_grid (Λ) ===")
for lam, rr in bound_grid(
    V, P, Q, base, param="Lambda", values=[1.0, 1.5, 3.0, 7.0], warm_start=True
):
    print(f"  Λ={lam:g}: [{rr.lower:.4f}, {rr.upper:.4f}]  w={rr.width:.4f}  tight={rr.tight}")
